# 🌐 Distributed Causal Engine: Telemetry → ML → Symbolic Reasoning → Recovery
Fixed for Kaggle: idempotent runs, GPU/CPU fallback, async→sync, ONNX parity, shape-safe tensors.

In [ ]:
# CELL 1: Setup & Dependencies
!pip install -q onnx onnxruntime onnxscript networkx scipy

import os, sys, json, time, asyncio, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, f1_score
import networkx as nx
from scipy.stats import expon, weibull_min
from typing import Dict, List, Tuple, AsyncIterator
from collections import deque
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
WORK_DIR = "/kaggle/working"

In [ ]:
# CELL 2: Component 1 → Realistic Synthetic Telemetry Generator
TOPOLOGY = {
    "gateway": ["lb-1", "lb-2"], "lb-1": ["auth-svc", "api-svc"], "lb-2": ["auth-svc", "api-svc"],
    "auth-svc": ["user-db", "cache"], "api-svc": ["rec-engine", "user-db"], "rec-engine": ["cache"],
    "user-db": [], "cache": []
}
NODE_TYPES = {"gateway": "edge", "lb-1": "lb", "lb-2": "lb", "auth-svc": "app", "api-svc": "app",
              "rec-engine": "app", "user-db": "db", "cache": "cache"}
DEPENDENTS = {n: [] for n in TOPOLOGY}
for src, dsts in TOPOLOGY.items():
    for dst in dsts: DEPENDENTS[dst].append(src)

SIM_DURATION_S = 86400
BASE_ARRIVAL_RATE = 200
QUEUE_SERVICE_RATE = 1000
BURSTINESS_HURST = 0.75

def generate_traffic(base_rate, duration, hurst):
    t = np.arange(1, duration + 1)
    window = int(duration ** hurst)
    noise = np.random.randn(duration)
    smoothed = np.convolve(noise, np.ones(window)/window, mode='same')
    return np.maximum(base_rate * (1 + 0.4 * smoothed), 10)

def compute_queue_metrics(arrival_rates, service_rate):
    rho = np.clip(arrival_rates / service_rate, 0.0, 0.99)
    queue_depth = rho / (1 - rho)
    queuing_delay = 1.0 / (service_rate - arrival_rates)
    return {"utilization": rho, "queue_depth": queue_depth, "queuing_delay_ms": queuing_delay * 1000}

def inject_failures(duration, prob_per_sec=0.002):
    times, types = [], []
    for _ in range(int(duration * prob_per_sec)):
        times.append(np.random.randint(0, duration))
        types.append(np.random.choice(["software_crash", "network_partition", "disk_saturation"], p=[0.6, 0.25, 0.15]))
    return pd.DataFrame({"time_s": times, "failure_type": types})

def propagate_effects(failures_df, duration, delay=1.5):
    effects = []
    for _, row in failures_df.iterrows():
        t0 = row["time_s"]
        affected = DEPENDENTS.get(row.get("node", "unknown"), [])
        for t in np.arange(t0 + delay, min(t0 + 10, duration), 0.5):
            for node in affected:
                effects.append({"time_s": t, "affected_node": node, "root_cause_time": t0})
    return pd.DataFrame(effects)

# Generate & Persist
telemetry = []
for node in TOPOLOGY:
    base = BASE_ARRIVAL_RATE * (1.5 if NODE_TYPES[node] == "gateway" else 1.0)
    arrivals = generate_traffic(base, SIM_DURATION_S, BURSTINESS_HURST)
    q = compute_queue_metrics(arrivals, QUEUE_SERVICE_RATE)
    df = pd.DataFrame({"timestamp": np.arange(SIM_DURATION_S), "node": node, "packets_in": arrivals, **q})
    telemetry.append(df)
telemetry_df = pd.concat(telemetry, ignore_index=True)

failures = inject_failures(SIM_DURATION_S)
forced = ["user-db", "auth-svc", "gateway"]
for i, n in enumerate(forced):
    if i < len(failures): failures.loc[i, "node"] = n
if len(failures) > len(forced):
    failures.loc[len(forced):, "node"] = np.random.choice(list(TOPOLOGY.keys()), len(failures)-len(forced))

cascade = propagate_effects(failures, SIM_DURATION_S)
telemetry_df["base_delay"] = telemetry_df["queuing_delay_ms"].copy()
modified = 0
for _, c in cascade.iterrows():
    mask = (telemetry_df["node"] == c["affected_node"]) & \
           (telemetry_df["timestamp"] >= int(c["time_s"])) & \
           (telemetry_df["timestamp"] <= int(c["time_s"]) + 5)
    telemetry_df.loc[mask, "queuing_delay_ms"] = telemetry_df.loc[mask, "base_delay"] * 3.0 + 2.5
    telemetry_df.loc[mask, "utilization"] = np.clip(telemetry_df.loc[mask, "utilization"] * 1.4, 0, 1)
    modified += mask.sum()
telemetry_df.drop(columns=["base_delay"], inplace=True)

telemetry_df.to_csv(f"{WORK_DIR}/network_telemetry_v1.csv", index=False)
failures.to_csv(f"{WORK_DIR}/failure_ground_truth.csv", index=False)
cascade.to_csv(f"{WORK_DIR}/cascade_events.csv", index=False)
print(f"✅ Telemetry: {len(telemetry_df)} rows | Failures: {len(failures)} | Cascade: {len(cascade)} | Modified: {modified}")

In [ ]:
# CELL 3: Component 2 → Windowing & Tensor Assembly
telemetry = pd.read_csv(f"{WORK_DIR}/network_telemetry_v1.csv")
ground_truth = pd.read_csv(f"{WORK_DIR}/failure_ground_truth.csv")
NODES = list(telemetry["node"].unique())
FEATURE_COLS = ["utilization", "queue_depth", "queuing_delay_ms", "packets_in"]
WINDOW_SIZE, STEP_SIZE, LOOKAHEAD = 10, 5, 5

windows, starts, node_ids = [], [], []
for idx, node in enumerate(NODES):
    ndf = telemetry[telemetry["node"] == node].sort_values("timestamp").reset_index(drop=True)
    for s in range(0, len(ndf) - WINDOW_SIZE + 1, STEP_SIZE):
        win = ndf.iloc[s:s+WINDOW_SIZE][FEATURE_COLS].values
        win_norm = (win - win.mean(axis=0)) / (win.std(axis=0) + 1e-8)
        windows.append(win_norm)
        starts.append(ndf.iloc[s]["timestamp"])
        node_ids.append(idx)
windows = np.array(windows)
starts = np.array(starts)
node_ids = np.array(node_ids)

# Causal Labels
labels = np.zeros(len(starts))
gt_by_node = ground_truth.groupby("node")["time_s"].apply(list).to_dict()
for i in range(len(starts)):
    node_name = NODES[node_ids[i]]
    if node_name in gt_by_node:
        for ft in gt_by_node[node_name]:
            if starts[i] <= ft <= starts[i] + WINDOW_SIZE + LOOKAHEAD:
                labels[i] = 1.0
                break

# Tensor Assembly
X_full = torch.FloatTensor(np.tile(windows[:, :, :, np.newaxis], (1, 1, 1, len(NODES))))
y_full = torch.FloatTensor(labels)
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)
print(f"✅ Tensors: {X_full.shape} | Positives: {y_full.sum():.0f} ({100*y_full.mean():.1f}%)")
print(f"📊 Train: {len(X_train)} | Val: {len(X_val)} | Batches/epoch: {len(train_loader)}")

In [ ]:
# CELL 4: Component 3 → Model, Training & ONNX Export
class TemporalGraphPredictor(nn.Module):
    def __init__(self, time_steps=10, features=4, nodes=8, hidden=32):
        super().__init__()
        self.conv = nn.Conv1d(features, hidden, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(hidden * nodes, hidden), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden, 1))
    def forward(self, x): # x: (B, T, F, N)
        B, T, F, N = x.shape
        x = x.permute(0, 3, 2, 1).reshape(B*N, F, T)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        x = x.reshape(B, -1)
        return self.fc(x).squeeze(-1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0): super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = probs * targets + (1 - probs) * (1 - targets)
        return (self.alpha * (1 - pt)**self.gamma * ce).mean()

model = TemporalGraphPredictor().to(device)
criterion = FocalLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

epochs, patience, best_val, trigger = 30, 5, float('inf'), 0
best_thresh = 0.5
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        out = model(Xb.to(device))
        loss = criterion(out, yb.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss, probs, lbls = 0.0, [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            out = model(Xb.to(device))
            val_loss += criterion(out, yb.to(device)).item()
            probs.append(torch.sigmoid(out).cpu())
            lbls.append(yb.cpu())
    val_loss /= len(val_loader)
    scheduler.step()
    
    p_flat = torch.cat(probs).numpy()
    l_flat = torch.cat(lbls).numpy()
    prec, rec, threshs = precision_recall_curve(l_flat, p_flat)
    f1 = 2*(prec*rec)/(prec+rec+1e-8)
    best_thresh = threshs[np.argmax(f1)] if len(threshs) > 0 else 0.5
    
    print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss/len(train_loader):.4f} | Val: {val_loss:.4f} | F1: {np.max(f1):.3f} | Thresh: {best_thresh:.2f}")
    if val_loss < best_val: best_val, trigger = val_loss, 0; torch.save(model.state_dict(), f"{WORK_DIR}/best_anomaly_predictor.pt")
    else: trigger += 1
    if trigger >= patience: print("⏹️ Early stopping"); break

# ONNX Export
model.load_state_dict(torch.load(f"{WORK_DIR}/best_anomaly_predictor.pt"))
model.eval()
dummy = torch.randn(1, 10, 4, 8).to(device)
torch.onnx.export(model, dummy, f"{WORK_DIR}/anomaly_predictor.onnx", input_names=["window"], output_names=["logit"],
                  dynamic_axes={"window": {0: "batch"}}, opset_version=17, do_constant_folding=True)
print(f"✅ ONNX exported: {os.path.getsize(f'{WORK_DIR}/anomaly_predictor.onnx')/1024:.1f} KB")

In [ ]:
# CELL 5: Component 4 → Symbolic Causal Engine
class DependencyGraph:
    def __init__(self, topology):
        self.graph = nx.DiGraph()
        for src, dsts in topology.items():
            self.graph.add_node(src, type=self._infer_type(src))
            for dst in dsts:
                self.graph.add_edge(src, dst, weight=1.0)
                self.graph.add_node(dst, type=self._infer_type(dst))
        self.metrics_history = {n: deque(maxlen=10) for n in self.graph.nodes}
    def _infer_type(self, n): return "data" if "db" in n or "cache" in n else ("edge" if "lb" in n or "gateway" in n else "service")
    def update_metrics(self, node, metrics): self.metrics_history[node].append(metrics)
    def get_recent_metrics(self, node): return {k: np.mean([m[k] for m in self.metrics_history[node]]) for k in self.metrics_history[node][0]} if self.metrics_history[node] else {}

class TemporalCausalityEngine:
    def __init__(self, max_delay=2.0, lat_thresh=3.0): self.max_delay, self.lat_thresh = max_delay, lat_thresh
    def isolate_root_cause(self, graph, suspects, current_time):
        results = []
        for sus in suspects:
            if sus not in graph.graph: continue
            m = graph.get_recent_metrics(sus)
            if m.get("queuing_delay_ms", 0) < self.lat_thresh: continue
            upstream = list(graph.graph.predecessors(sus))
            is_root = True
            for up in upstream:
                if up in suspects:
                    um = graph.get_recent_metrics(up)
                    if um.get("queuing_delay_ms", 0) >= m.get("queuing_delay_ms", 0): is_root = False; break
            if is_root:
                blast = list(nx.descendants(graph.graph, sus))
                results.append({"root_node": sus, "confidence": min(1.0, m.get("queuing_delay_ms", 0)/10.0), "blast_radius": blast, "temporal_window": self.max_delay})
        return results

class RecoveryPlanner:
    def __init__(self):
        self.templates = {"data": ["enable_read_replica", "flush_pool", "restart"], "service": ["scale_out", "circuit_breaker", "drain"], "edge": ["reroute", "dns_ttl", "rate_limit"]}
    def generate_plan(self, root, graph):
        ntype = graph.graph.nodes[root].get("type", "service")
        return [{"step": i+1, "action": a, "target": root, "wait_s": 2} for i, a in enumerate(self.templates.get(ntype, self.templates["service"]))]

print("✅ Causal Engine Ready")

In [ ]:
# CELL 6: Component 5 → Live Pipeline Integration & Validation
import onnxruntime as ort

class SymbolicCausalPipeline:
    def __init__(self, graph, onnx_path, failure_thresh=0.35, delay_thresh=3.0, util_thresh=0.85):
        self.graph = graph
        self.causal = TemporalCausalityEngine(lat_thresh=delay_thresh)
        self.sess = ort.InferenceSession(onnx_path)
        self.iname = self.sess.get_inputs()[0].name
        self.thresh, self.dthresh, self.uthresh = failure_thresh, delay_thresh, util_thresh
    def ingest_window(self, window_tensor, timestamp, debug=False, node_order=None):
        nodes = node_order or list(self.graph.graph.nodes())
        for i, node in enumerate(nodes):
            self.graph.update_metrics(node, {k: float(window_tensor[0, -1, j, i]) for j, k in enumerate(["utilization", "queue_depth", "queuing_delay_ms", "packets_in"])})
        suspects = [n for n in nodes if self.graph.get_recent_metrics(n).get("queuing_delay_ms", 0) > self.dthresh or self.graph.get_recent_metrics(n).get("utilization", 0) > self.uthresh]
        if not suspects: return []
        return self.causal.isolate_root_cause(self.graph, suspects, timestamp)

class StreamingWindowBuffer:
    def __init__(self, window_size=10, feature_cols=None, node_order=None):
        self.ws = window_size
        self.fcols = feature_cols or ["utilization", "queue_depth", "queuing_delay_ms", "packets_in"]
        self.node_order = node_order
        self.buffers = {n: deque(maxlen=self.ws) for n in self.node_order}
        self.ready = []
    def ingest(self, node, metrics):
        self.buffers[node].append(metrics)
        if all(len(b) == self.ws for b in self.buffers.values()):
            win = np.zeros((1, self.ws, len(self.fcols), len(self.node_order)), dtype=np.float32)
            for i, n in enumerate(self.node_order):
                for j, m in enumerate(self.buffers[n]):
                    for k, c in enumerate(self.fcols): win[0, j, k, i] = m[c]
            self.ready.append(win)
            for b in self.buffers.values(): b.clear()
    def pop_ready(self): w = self.ready.copy(); self.ready.clear(); return w

# Sync Validation Loop
graph = DependencyGraph(TOPOLOGY)
planner = RecoveryPlanner()
pipeline = SymbolicCausalPipeline(graph, f"{WORK_DIR}/anomaly_predictor.onnx", failure_threshold=0.30, delay_thresh=3.0, util_thresh=0.85)
buffer = StreamingWindowBuffer(window_size=10, node_order=list(graph.graph.nodes()))

# Simulate stream from CSV
telemetry = pd.read_csv(f"{WORK_DIR}/network_telemetry_v1.csv").sort_values("timestamp")
gt = pd.read_csv(f"{WORK_DIR}/failure_ground_truth.csv")
detections, ground_truth_log = [], []
start_time = time.time()

for _, row in telemetry.iterrows():
    t = row["timestamp"]
    node = row["node"]
    metrics = {k: row[k] for k in ["utilization", "queue_depth", "queuing_delay_ms", "packets_in"]}
    buffer.ingest(node, metrics)
    for win in buffer.pop_ready():
        chains = pipeline.ingest_window(win, t, node_order=list(graph.graph.nodes()))
        if chains:
            chain = chains[0]
            plan = planner.generate_plan(chain["root_node"], graph)
            matched = any(g["node"] == chain["root_node"] or g["node"] in chain["blast_radius"] for _, g in gt.iterrows())
            detections.append({"root": chain["root_node"], "conf": chain["confidence"], "matched_gt": matched, "latency_s": time.time()-start_time})
            if len(detections) >= 3: break
    if len(detections) >= 3: break

print("\n" + "="*60)
print("📊 LIVE PIPELINE VALIDATION REPORT")
print("="*60)
print(f"⚡ Detections: {len(detections)}")
if detections:
    print(f"✅ GT Match Rate: {sum(1 for d in detections if d['matched_gt'])/len(detections):.0%}")
    print(f"⏱️ Avg Latency: {np.mean([d['latency_s'] for d in detections]):.2f}s")
print("✅ PIPELINE COMPLETE")

In [ ]:
# CELL 7: Export & Local Integration Hook
torch.save({"tensor_shape": X_full.shape, "features": FEATURE_COLS, "nodes": NODES, "window": {"size": WINDOW_SIZE, "step": STEP_SIZE}}, f"{WORK_DIR}/component2_metadata.pt")
print("📦 Exported Artifacts:")
print(f" - {WORK_DIR}/network_telemetry_v1.csv")
print(f" - {WORK_DIR}/failure_ground_truth.csv")
print(f" - {WORK_DIR}/best_anomaly_predictor.pt")
print(f" - {WORK_DIR}/anomaly_predictor.onnx")
print(f" - {WORK_DIR}/component2_metadata.pt")

local_hook = '''
# local_inference.py
import onnxruntime as ort
import numpy as np
class AnomalyPredictor:
    def __init__(self, path="anomaly_predictor.onnx"):
        self.sess = ort.InferenceSession(path)
        self.iname = self.sess.get_inputs()[0].name
    def predict(self, window_tensor): # shape (1, 10, 4, 8)
        logit = self.sess.run(None, {self.iname: window_tensor})[0][0]
        return 1 / (1 + np.exp(-logit))
'''
print("\n🔌 Local Integration Template:")
print(local_hook)